In [ ]:
data = [
    ['Sunny', 'Warm', 'Normal', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Yes'],
    ['Rainy', 'Cold', 'High', 'No'],
    ['Sunny', 'Warm', 'High', 'Yes']
]

hypothesis = ['0'] * (len(data[0]) - 1)

for row in data:
    if row[-1] == 'Yes':
        if hypothesis[0] == '0':
            hypothesis = row[:-1]
        else:
            for i in range(len(hypothesis)):
                if hypothesis[i] != row[i]:
                    hypothesis[i] = '?'

print("Final Hypothesis:", hypothesis)


Final Hypothesis: ['Sunny', 'Warm', '?']


In [ ]:
def candidate_elimination(data):
    attributes = [row[:-1] for row in data]
    targets = [row[-1] for row in data]

    num_attributes = len(attributes[0])
    S = [['0'] * num_attributes]
    G = [['?' for _ in range(num_attributes)]]

    for i, instance in enumerate(attributes):
        if targets[i] == 'Yes':
            G = [g for g in G if all(g[j] == '?' or g[j] == instance[j] for j in range(num_attributes))]
            for s in S:
                for j in range(num_attributes):
                    if s[j] == '0':
                        s[j] = instance[j]
                    elif s[j] != instance[j]:
                        s[j] = '?'
        else:
            S = [s for s in S if not all(s[j] == '?' or s[j] == instance[j] for j in range(num_attributes))]
            new_G = []
            for g in G:
                for j in range(num_attributes):
                    if g[j] == '?':
                        for val in set(attr[j] for attr in attributes):
                            if val != instance[j]:
                                new_hypothesis = g.copy()
                                new_hypothesis[j] = val
                                new_G.append(new_hypothesis)
            G = new_G

    return S, G


training_data = [
    ['Sunny', 'Warm', 'Normal', 'Yes'],
    ['Sunny', 'Warm', 'High', 'Yes'],
    ['Rainy', 'Cold', 'High', 'No'],
    ['Sunny', 'Cold', 'High', 'Yes']
]

S, G = candidate_elimination(training_data)
print("Specific boundary (S):", S)
print("General boundary (G):", G)


Specific boundary (S): [['Sunny', '?', '?']]
General boundary (G): [['Sunny', '?', '?']]


In [ ]:
import math

def entropy(examples):
    pos = sum(1 for e in examples if e[-1] == 'Yes')
    neg = sum(1 for e in examples if e[-1] == 'No')
    total = len(examples)
    if pos == 0 or neg == 0:
        return 0
    p_pos = pos / total
    p_neg = neg / total
    return -p_pos * math.log2(p_pos) - p_neg * math.log2(p_neg)

def information_gain(examples, attr_index):
    total_entropy = entropy(examples)
    values = set(e[attr_index] for e in examples)
    weighted_entropy = 0
    for v in values:
        subset = [e for e in examples if e[attr_index] == v]
        weighted_entropy += (len(subset) / len(examples)) * entropy(subset)
    return total_entropy - weighted_entropy

def id3(examples, attributes):
    targets = [e[-1] for e in examples]
    if targets.count(targets[0]) == len(targets):
        return targets[0]
    if not attributes:
        return max(set(targets), key=targets.count)

    gains = [information_gain(examples, i) for i in attributes]
    best_attr = attributes[gains.index(max(gains))]
    tree = {best_attr: {}}
    values = set(e[best_attr] for e in examples)
    for v in values:
        subset = [e for e in examples if e[best_attr] == v]
        if not subset:
            tree[best_attr][v] = max(set(targets), key=targets.count)
        else:
            new_attrs = [a for a in attributes if a != best_attr]
            tree[best_attr][v] = id3(subset, new_attrs)
    return tree

def classify(tree, sample):
    if isinstance(tree, str):
        return tree
    attr = next(iter(tree))
    value = sample[attr]
    if value in tree[attr]:
        return classify(tree[attr][value], sample)
    else:
        return "Unknown"

data = [
    ['Sunny', 'Hot', 'High', 'Weak', 'No'],
    ['Sunny', 'Hot', 'High', 'Strong', 'No'],
    ['Overcast', 'Hot', 'High', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Cool', 'Normal', 'Strong', 'No'],
    ['Overcast', 'Cool', 'Normal', 'Strong', 'Yes'],
    ['Sunny', 'Mild', 'High', 'Weak', 'No'],
    ['Sunny', 'Cool', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'Normal', 'Weak', 'Yes'],
    ['Sunny', 'Mild', 'Normal', 'Strong', 'Yes'],
    ['Overcast', 'Mild', 'High', 'Strong', 'Yes'],
    ['Overcast', 'Hot', 'Normal', 'Weak', 'Yes'],
    ['Rain', 'Mild', 'High', 'Strong', 'No']
]

attributes = list(range(len(data[0]) - 1))
tree = id3(data, attributes)

print("Decision Tree:", tree)

# Use integer keys for attributes
new_sample = {0: 'Sunny', 1: 'Cool', 2: 'High', 3: 'Strong'}
print("Classification of new sample:", classify(tree, new_sample))


Decision Tree: {0: {'Sunny': {2: {'High': 'No', 'Normal': 'Yes'}}, 'Rain': {3: {'Weak': 'Yes', 'Strong': 'No'}}, 'Overcast': 'Yes'}}
Classification of new sample: No


In [19]:
import numpy as np

# Sigmoid activation and derivative
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

# Training data (XOR truth table)
X = np.array([[0,0],
              [0,1],
              [1,0],
              [1,1]])

y = np.array([[0],
              [1],
              [1],
              [0]])

# Initialize weights and biases
np.random.seed(42)
input_layer_neurons = 2
hidden_layer_neurons = 2
output_neurons = 1

W_hidden = np.random.uniform(size=(input_layer_neurons, hidden_layer_neurons))
b_hidden = np.random.uniform(size=(1, hidden_layer_neurons))
W_output = np.random.uniform(size=(hidden_layer_neurons, output_neurons))
b_output = np.random.uniform(size=(1, output_neurons))

# Training with Backpropagation
epochs = 10000
learning_rate = 0.1

for _ in range(epochs):
    # Forward pass
    hidden_input = np.dot(X, W_hidden) + b_hidden
    hidden_output = sigmoid(hidden_input)

    final_input = np.dot(hidden_output, W_output) + b_output
    final_output = sigmoid(final_input)

    # Backpropagation
    error = y - final_output
    d_output = error * sigmoid_derivative(final_output)

    error_hidden = d_output.dot(W_output.T)
    d_hidden = error_hidden * sigmoid_derivative(hidden_output)

    # Update weights and biases
    W_output += hidden_output.T.dot(d_output) * learning_rate
    b_output += np.sum(d_output, axis=0, keepdims=True) * learning_rate
    W_hidden += X.T.dot(d_hidden) * learning_rate
    b_hidden += np.sum(d_hidden, axis=0, keepdims=True) * learning_rate

# Testing
print("Final outputs after training:")
print(final_output.round(3))

# Classify a new sample
new_sample = np.array([[1, 0]])  # XOR(1,0) should be 1
hidden_input = np.dot(new_sample, W_hidden) + b_hidden
hidden_output = sigmoid(hidden_input)
final_input = np.dot(hidden_output, W_output) + b_output
prediction = sigmoid(final_input)
print("Prediction for [1,0]:", prediction.round(3))


Final outputs after training:
[[0.053]
 [0.952]
 [0.952]
 [0.052]]
Prediction for [1,0]: [[0.952]]


In [20]:
import numpy as np
from collections import Counter

def euclidean_distance(x1, x2):
    return np.sqrt(np.sum((x1 - x2) ** 2))

class KNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X_train = X
        self.y_train = y

    def predict(self, X):
        predictions = [self._predict(x) for x in X]
        return np.array(predictions)

    def _predict(self, x):
        distances = [euclidean_distance(x, x_train) for x_train in self.X_train]
        k_indices = np.argsort(distances)[:self.k]
        k_nearest_labels = [self.y_train[i] for i in k_indices]
        most_common = Counter(k_nearest_labels).most_common(1)
        return most_common[0][0]

# Example dataset (points in 2D space with class labels)
X_train = np.array([
    [1, 2], [2, 3], [3, 3],
    [6, 5], [7, 7], [8, 6]
])
y_train = np.array([
    'ClassA', 'ClassA', 'ClassA',
    'ClassB', 'ClassB', 'ClassB'
])

# Train KNN
knn = KNN(k=3)
knn.fit(X_train, y_train)

# Test on new samples
X_test = np.array([[5, 5], [2, 2]])
predictions = knn.predict(X_test)

print("Predictions:", predictions)


Predictions: ['ClassB' 'ClassA']


In [21]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.datasets import load_iris

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train Naïve Bayes classifier
model = GaussianNB()
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate results
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy:", accuracy_score(y_test, y_pred))


Confusion Matrix:
[[19  0  0]
 [ 0 12  1]
 [ 0  0 13]]

Accuracy: 0.9777777777777777


In [22]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, accuracy_score

# Load dataset
iris = load_iris()
X = iris.data
y = iris.target

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Train Logistic Regression model
model = LogisticRegression(max_iter=200)
model.fit(X_train, y_train)

# Predict on test set
y_pred = model.predict(X_test)

# Evaluate results
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nAccuracy:", accuracy_score(y_test, y_pred))


Confusion Matrix:
[[19  0  0]
 [ 0 13  0]
 [ 0  0 13]]

Accuracy: 1.0
